# Linear Y rotations

Preserve six control qubits, three targets, slopes (2.1, 1, 7.0) and offsets (0.1, 0.3, 0.33). Target j rotates by slope[j]*x + offset[j]. The source prepares x=0; the supplemental coherent case probes every control value and a relative phase. Run every cell from the repository root. See WORKFLOW.md for acceptance and scope.


In [ ]:
import math

import flagquantum as fq
from flagquantum import synth
from flagquantum.synth import (
    PHASE,
    Output,
    QArray,
    QBit,
    R,
    X,
    allocate,
    control,
    hadamard_transform,
    qfunc,
)

SLOPES = (2.1, 1.0, 7.0)
OFFSETS = (0.1, 0.3, 0.33)


@qfunc
def rotate_if_one(angle: float, selector: QBit, target: QBit) -> None:
    control(selector, lambda: R(angle, math.pi / 2, target))


@qfunc
def main(
    angles: list[float],
    offsets: list[float],
    input_bits: list[float],
    coherent: int,
    x: Output[QArray[QBit]],
    ind: Output[QArray[QBit]],
) -> None:
    allocate(6, x)
    allocate(3, ind)
    for bit in range(6):
        if input_bits[bit] != 0:
            X(x[bit])
    if coherent == 1:
        hadamard_transform(x)
        PHASE(math.pi, x[0])
    for target in range(3):
        R(offsets[target], math.pi / 2, ind[target])
        for bit in range(6):
            rotate_if_one(angles[6 * target + bit], x[bit], ind[target])



In [ ]:
binding = {
    "angles": [slope * (1 << bit) for slope in SLOPES for bit in range(6)],
    "offsets": list(OFFSETS),
    "input_bits": [0.0] * 6,
    "coherent": 0,
}
program = synth.synthesize(main, classical_inputs=binding, max_workspace=0)
result = fq.run(program)
print(dict(program.probabilities(result, registers=("x", "ind"))))
print(program.report["resources"])


In [ ]:
coherent_binding = dict(binding, coherent=1)
coherent_program = synth.synthesize(main, classical_inputs=coherent_binding, max_workspace=0)
coherent_result = fq.run(coherent_program)
print(dict(coherent_program.probabilities(coherent_result, registers=("x", "ind"))))
